# NB03: Core vs. Accessory Enrichment and Phylogenetic Stratification

Tests whether co-occurring nutrient and metal genes concentrate in the core genome,
and stratifies co-occurrence by GTDB taxonomy.

Run equivalent scripts: `python src/03_core_accessory_enrichment.py` and `python src/04_phylogenetic_stratification.py`

In [1]:
import pandas as pd
import numpy as np
import os
DATA_DIR = os.path.join('..', 'data')

## Core vs. Accessory Enrichment

In [2]:
summary = pd.read_csv(os.path.join(DATA_DIR, 'core_enrichment_summary.csv'))
summary

,nutrient_group,n_species_both,n_species_testable,nutrient_core_frac,metal_core_frac,stouffer_z,stouffer_p,median_OR,frac_enriched
0,P_genes,20683,15600,0.7071,0.5122,68.304,0.000000,NaN,0.300
1,N_genes,5718,2549,0.4614,0.4459,-4.853,0.000001,NaN,0.144
2,Phz_genes,10632,3796,0.6548,0.5329,2.636,0.008392,NaN,0.175


In [3]:
df = pd.read_csv(os.path.join(DATA_DIR, 'species_gene_families.csv'))
all_genes = ['phoA','phoD_pfam','pstA','pstC','pstS','phnC','phnE',
             'nifH','nifD','nifH_pfam','phzF','phzG','phzS',
             'copA','corA','feoB_pfam','HMA_pfam']
print('Gene family core fractions:')
for g in all_genes:
    total = int(df[f'n_{g}'].sum())
    core = int(df[f'n_{g}_core'].sum())
    frac = core/total if total > 0 else 0
    print(f'  {g:>12s}: {frac:.3f} ({core}/{total})')

Gene family core fractions:
          phoA: 0.627 (6521/10394)
     phoD_pfam: 0.172 (285/1661)
          pstA: 0.737 (17864/24249)
          pstC: 0.765 (14150/18507)
          pstS: 0.735 (10015/13626)
          phnC: 0.652 (4954/7593)
          phnE: 0.631 (9720/15394)
          nifH: 0.649 (2203/3396)
          nifD: 0.609 (1034/1699)
     nifH_pfam: 0.324 (2152/6632)
          phzF: 0.653 (11026/16882)
          phzG: 0.533 (73/137)
          phzS: 0.659 (1121/1700)
          copA: 0.568 (17595/30994)
          corA: 0.735 (18094/24605)
     feoB_pfam: 0.300 (5896/19684)
      HMA_pfam: 0.236 (2746/11648)


## Phylum-Level Stratification

In [4]:
phy = pd.read_csv(os.path.join(DATA_DIR, 'phylum_cooccurrence.csv'))
top = phy[phy['n_species']>=50].sort_values('phi_PM', ascending=False).head(12)
top[['phylum','n_species','frac_P','frac_M','phi_PM','PM_enrichment','n_Phz_operon']]

,phylum,n_species,frac_P,frac_M,phi_PM,PM_enrichment,n_Phz_operon
29,p__Fusobacteriota,59,0.881,0.915,0.4530,1.051,0
7,p__Cyanobacteriota,469,0.823,0.736,0.3554,1.099,0
4,p__Bacillota,2146,0.836,0.904,0.2456,1.035,0
26,p__Bdellovibrionota,71,0.915,0.958,0.1879,1.012,0
21,p__Gemmatimonadota,102,0.892,0.980,0.1788,1.009,0
23,p__Marinisomatota,90,0.467,0.778,0.1786,1.102,0
24,p__Omnitrophota,80,0.887,0.762,0.1731,1.034,0
0,p__Pseudomonadota,7456,0.909,0.895,0.1672,1.018,27
9,p__Planctomycetota,348,0.931,0.899,0.1352,1.012,0
14,p__Campylobacterota,271,0.782,0.889,0.0988,1.018,0


## Phenazine Operon Taxonomy

In [5]:
phz = pd.read_csv(os.path.join(DATA_DIR, 'phenazine_operon_taxonomy.csv'))
print(f'{len(phz)} phenazine operon species')
print()
print('By family:')
for fam, grp in phz.groupby('family'):
    genera = grp['genus'].unique()
    print(f'  {fam}: {len(grp)} species ({', '.join(genera[:4])})')

63 phenazine operon species

By family:
  f__Burkholderiaceae: 1 species (g__Burkholderia)
  f__Enterobacteriaceae: 6 species (g__Xenorhabdus)
  f__Jiangellaceae: 1 species (g__Jiangella)
  f__Pseudomonadaceae: 17 species (g__Pseudomonas_E, g__Pseudomonas)
  f__Pseudonocardiaceae: 1 species (g__Saccharopolyspora)
  f__SG8-39: 1 species (g__CADEDU01)
  f__Streptomycetaceae: 24 species (g__Streptomyces, g__Kitasatospora)
  f__Streptosporangiaceae: 10 species (g__Microbispora, g__Nocardiopsis, g__Spirillospora, g__Nonomuraea)
  f__Xanthomonadaceae: 2 species (g__Lysobacter)


## Plant-Associated Lineages

In [6]:
tax = pd.read_csv(os.path.join(DATA_DIR, 'species_taxonomy.csv'))
merged = df.merge(tax[['gtdb_species_clade_id','family']], on='gtdb_species_clade_id', how='left')

plant_fams = ['f__Pseudomonadaceae','f__Rhizobiaceae','f__Burkholderiaceae',
              'f__Streptomycetaceae','f__Xanthomonadaceae','f__Enterobacteriaceae']
for fam in plant_fams:
    s = merged[merged['family']==fam]
    if len(s)==0: continue
    n=len(s); np_=int(s['has_P_acquisition'].sum()); nm=int(s['has_metal_handling'].sum())
    nphz=int(s['has_phz_operon'].sum())
    print(f'{fam}: n={n} P={np_/n:.2f} M={nm/n:.2f} Phz_operon={nphz}')

f__Pseudomonadaceae: n=498 P=1.00 M=1.00 Phz_operon=17
f__Rhizobiaceae: n=413 P=1.00 M=0.93 Phz_operon=0
f__Burkholderiaceae: n=333 P=1.00 M=0.99 Phz_operon=1
f__Streptomycetaceae: n=382 P=1.00 M=1.00 Phz_operon=24
f__Xanthomonadaceae: n=207 P=1.00 M=0.98 Phz_operon=2
f__Enterobacteriaceae: n=472 P=0.89 M=0.96 Phz_operon=6
